In [1]:
# blastn if no data
import os
from Bio import SeqIO
import pandas as pd

genus_name = 'Escherichia'
max_name = 'GCF_022569795.1'
acc_n = max_name

handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
acc_record = SeqIO.parse(handle, 'genbank')
data_folder = f'/active-data/analysis_results/chr_pla/genus/max_matched/{genus_name}/{acc_n}'
if not os.path.exists(data_folder):
    os.makedirs(data_folder)
for seq_record in acc_record:
    seq_len = len(seq_record)
    blast_seq = seq_record
    os.chdir(data_folder)
    print(seq_record.id)
    blast_result_file = f'blastn_results_{acc_n}_{seq_record.id}.txt'
    if os.path.exists(blast_result_file):
        continue
    temp_ncl_file = open(f'temp_nucleotide_seq_{acc_n}.fasta', 'w+')
    SeqIO.write(blast_seq, temp_ncl_file, "fasta")
    temp_ncl_file.close()
    %time os.system(f'blastn -query temp_nucleotide_seq_{acc_n}.fasta -db /active-data/analysis_results/chr_pla/genus/blast_data/{genus_name}/nucleotide_seq.blastdb -out {blast_result_file} -evalue 1e-50 -max_target_seqs 100000 -outfmt 6 -num_threads 8')
    #head = ['qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen', 'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore']
    #align_result = pd.read_csv(f'blastn_results_{acc_n}.txt', sep = '\t|;', engine = 'python', header = None, names = head)

NZ_CP093011.1


In [2]:
head = ['qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen', 'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore']
align_result = pd.read_csv(f'{data_folder}/blastn_results_{acc_n}_{seq_record.id}.txt', sep = '\t|;', engine = 'python', header = None, names = head)

folder = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}'
os.chdir(folder)

stat_file = "replicon-plasmid_fraction-self_bitscore_statistics.csv"
stat_df = pd.read_csv(stat_file)

link_file = f"{genus_name}_typical_chromosome_sub_replicon_link.csv"
link_df = pd.read_csv(link_file)
filted_link_df = link_df[link_df['target'] == max_name]

trasi_r = stat_df[(stat_df['category-pident_90'] == 'intermediate replicon')]['accession'].to_list()
typical_p = stat_df[(stat_df['category-pident_90'] == 'typical plasmid')]['accession'].to_list()

print(len(trasi_r), len(typical_p))

153 11079


In [3]:
import numpy as np

def coverage_stat(df, temp_result, seq_len):
    coverage = np.zeros(int(seq_len), dtype=int)
    for order, (sid, score) in enumerate(zip(df['accession'], df['average plasmid fraction-pident_90'])):
        df_sub = temp_result[temp_result['sseqid'] == sid]
        intervals = df_sub[["qstart", "qend"]].values.tolist()
        intervals.sort()
    
        merged = []
        for start, end in intervals:
            if not merged:
                merged.append([start, end])
            else:
                last_start, last_end = merged[-1]
                if start <= last_end:
                    new_start = last_start
                    new_end = max(last_end, end)
                    merged[-1] = [new_start, new_end]
                else:
                    merged.append([start, end])
                    
        for region in merged:
            start = max(0, region[0] - 1)
            end = max(0, region[1] - 1)
            if start > end:
                continue
            coverage[start:end+1] += 1

    return coverage

for filted in [True, False]:
    if filted:
        covered_replicons = filted_link_df['source'].unique()
        suffix = '_filted'
    else:
        covered_replicons = trasi_r + typical_p
        suffix = ''

    temp_result = align_result[(align_result['sseqid'].isin(covered_replicons)) & (align_result['pident'] >= 90)]
    
    unique_ids = temp_result['sseqid'].unique()
    
    id_score_df = stat_df[stat_df['accession'].isin(unique_ids)][['accession', 'average plasmid fraction-pident_90']].copy()
    id_score_df = id_score_df.sort_values(by='average plasmid fraction-pident_90', ascending=True).reset_index(drop=True)
    
    sorted_id_list = id_score_df['accession'].tolist()
    sorted_score_list = id_score_df['average plasmid fraction-pident_90'].tolist()
    
    transi_score_df = id_score_df[id_score_df["average plasmid fraction-pident_90"] < 0.3]
    typical_score_df = id_score_df[id_score_df["average plasmid fraction-pident_90"] >= 0.3]
    print(len(transi_score_df))
    
    %time trans_coverage = coverage_stat(transi_score_df, temp_result, seq_len)
    %time typical_coverage = coverage_stat(typical_score_df, temp_result, seq_len)
    print(max(trans_coverage), max(typical_coverage))
    
    result_dir = f'/active-data/analysis_results/chr_pla/genus/figure_data/example_coverage/{genus_name}'
    os.makedirs(result_dir, exist_ok=True)
    os.chdir(result_dir)
    
    np.savetxt(f"{max_name}_trans_coverage{suffix}.txt", trans_coverage, fmt='%d')
    np.savetxt(f"{max_name}_typical_coverage{suffix}.txt", typical_coverage, fmt='%d')
    id_score_df.to_csv(f"{max_name}_id_score_df{suffix}.csv", index=False)

48
CPU times: user 245 ms, sys: 6.92 ms, total: 252 ms
Wall time: 252 ms
CPU times: user 7.44 s, sys: 8.52 ms, total: 7.44 s
Wall time: 7.45 s
8 757
112
CPU times: user 3.19 s, sys: 14.8 ms, total: 3.2 s
Wall time: 3.21 s
CPU times: user 4min 6s, sys: 480 ms, total: 4min 6s
Wall time: 4min 7s
14 4369
